# Erasus — VLM Coreset Ablation (CLIP ViT-B/32, GPU)
Coreset-based unlearning on CLIP. Fine-tune on CIFAR-10, unlearn airplane+automobile classes.

In [ ]:
!pip install -q git+https://github.com/OnePunchMonk/erasus.git@docs/rewrite-readme accelerate diffusers ftfy matplotlib open_clip_torch
import torch
print(f"CUDA: {torch.cuda.is_available()}, Device: {torch.cuda.get_device_name(0)}")

In [ ]:
import copy, time, json, torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader, TensorDataset, Subset
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
device = torch.device('cuda')

import open_clip
clip_model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
clip_model = clip_model.to(device).float()
n_params = sum(p.numel() for p in clip_model.parameters())
print(f'CLIP ViT-B/32: {n_params/1e6:.0f}M parameters')

cifar_transform = transforms.Compose([
    transforms.Resize(224), transforms.ToTensor(),
    transforms.Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)),
])
full_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=cifar_transform)
np.random.seed(42)
forget_indices = [i for i, (_, y) in enumerate(full_train) if y in [0, 1]][:500]
retain_indices = [i for i, (_, y) in enumerate(full_train) if y not in [0, 1]][:1500]
forget_set = Subset(full_train, forget_indices)
retain_set = Subset(full_train, retain_indices)
forget_loader = DataLoader(forget_set, batch_size=32, shuffle=False)
retain_loader = DataLoader(retain_set, batch_size=32, shuffle=False)
train_loader = DataLoader(Subset(full_train, forget_indices + retain_indices), batch_size=32, shuffle=True)
print(f'Forget: {len(forget_set)}, Retain: {len(retain_set)}')

In [ ]:
class CLIPClassifier(nn.Module):
    def __init__(self, clip_model, n_classes=10):
        super().__init__()
        self.clip = clip_model
        self.classifier = nn.Linear(512, n_classes).to(device)
        for p in self.clip.transformer.parameters(): p.requires_grad = False
    def forward(self, x, labels=None, **kwargs):
        with torch.no_grad() if not self.training else torch.enable_grad():
            features = self.clip.encode_image(x)
        features = F.normalize(features.float(), dim=-1)
        logits = self.classifier(features)
        if labels is not None:
            return type('Out', (), {'logits': logits, 'loss': F.cross_entropy(logits, labels)})()
        return logits

model = CLIPClassifier(clip_model, n_classes=10)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
model.train()
for epoch in range(10):
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        out = model(x, labels=y); out.loss.backward()
        optimizer.step(); optimizer.zero_grad(); total_loss += out.loss.item()
    if (epoch+1)%5==0: print(f'Epoch {epoch+1}/10 Loss: {total_loss/len(train_loader):.4f}')

@torch.no_grad()
def compute_accuracy(model, loader):
    model.eval(); correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x); logits = out.logits if hasattr(out,'logits') else out
        correct += (logits.argmax(-1)==y).sum().item(); total += y.size(0)
    return correct/max(total,1)

base_forget_acc = compute_accuracy(model, forget_loader)
base_retain_acc = compute_accuracy(model, retain_loader)
print(f'Base: Forget Acc={base_forget_acc:.4f}, Retain Acc={base_retain_acc:.4f}')
trained_state = copy.deepcopy(model.state_dict())

In [ ]:
import erasus.strategies; import erasus.selectors
from erasus.unlearners import ErasusUnlearner
FRACTIONS = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
SELECTORS = ['gradient_norm', 'el2n', 'herding', 'kcenter', 'random']
vlm_results = {}
for sel in SELECTORS:
    print(f'  {sel}: ', end='', flush=True); sel_results = []
    for frac in FRACTIONS:
        m = CLIPClassifier(clip_model, n_classes=10)
        m.load_state_dict(copy.deepcopy(trained_state))
        try:
            unlearner = ErasusUnlearner(model=m, strategy='gradient_ascent',
                selector=sel if frac<1.0 else None, device='cuda', strategy_kwargs={'lr':1e-4})
            result = unlearner.fit(forget_data=forget_loader, retain_data=retain_loader,
                prune_ratio=frac if frac<1.0 else None, epochs=5)
            fa = compute_accuracy(unlearner.model, forget_loader)
            ra = compute_accuracy(unlearner.model, retain_loader)
            sel_results.append({'fraction':frac,'forget_accuracy':round(fa,4),'retain_accuracy':round(ra,4),'status':'OK'})
            print(f'{frac*100:.0f}%\u2713 ', end='', flush=True)
        except Exception as e:
            sel_results.append({'fraction':frac,'forget_accuracy':None,'retain_accuracy':None,'status':'ERROR','error':str(e)[:100]})
            print(f'{frac*100:.0f}%\u2717 ', end='', flush=True)
        del m; torch.cuda.empty_cache()
    vlm_results[sel] = sel_results; print()

for sel, pts in vlm_results.items():
    for p in pts:
        if p['status']=='OK':
            print(f"  {sel:<16} {p['fraction']*100:>5.0f}% F.Acc:{p['forget_accuracy']:.4f} R.Acc:{p['retain_accuracy']:.4f}")

In [ ]:
colors = {'gradient_norm':'#377eb8','el2n':'#4daf4a','herding':'#ff7f00','kcenter':'#a65628','random':'#999'}
fig, axes = plt.subplots(1, 3, figsize=(18,6))
for sel, pts in vlm_results.items():
    ok = [p for p in pts if p['status']=='OK']
    fracs=[p['fraction'] for p in ok]; fa=[p['forget_accuracy'] for p in ok]
    ra=[p['retain_accuracy'] for p in ok]; sc=[(1-p['forget_accuracy'])*p['retain_accuracy'] for p in ok]
    ls='--' if sel=='random' else '-'
    axes[0].plot(fracs,fa,f'o{ls}',color=colors.get(sel,'#999'),label=sel)
    axes[1].plot(fracs,ra,f'o{ls}',color=colors.get(sel,'#999'),label=sel)
    axes[2].plot(fracs,sc,f'o{ls}',color=colors.get(sel,'#999'),label=sel)
for ax in axes: ax.legend(fontsize=8); ax.grid(True,alpha=0.3); ax.set_xlabel('Coreset Fraction')
axes[0].set_title('Forgetting Quality',fontweight='bold'); axes[0].set_ylabel('Forget Acc (lower=better)')
axes[1].set_title('Utility Preservation',fontweight='bold'); axes[1].set_ylabel('Retain Acc (higher=better)')
axes[2].set_title('Combined Score',fontweight='bold'); axes[2].set_ylabel('Score (higher=better)')
fig.suptitle('VLM Coreset Unlearning - CLIP ViT-B/32 (T4 GPU)',fontsize=14,fontweight='bold',y=1.02)
fig.tight_layout(); plt.show()